# 02 — Messi Era: Offside Trap Mechanism

**Question:** How did Barcelona's offside trap work during the Messi era (2004/05–2020/21), and what conditions predicted its success or failure?

**Central hypothesis:** The trap is only safe when coordinated with high pressing. A high defensive line without the press exposes space that cannot be recovered. If true, we expect a positive correlation between pressing intensity (PPDA) and offsides drawn, and a negative correlation between low-pressing matches and goals conceded behind the line.

**Plan for this notebook:**
1. Assign manager eras to each match
2. Compute offsides drawn per match (from `Pass Offside` events)
3. Compute PPDA per match (pressing intensity)
4. Compute defensive line height per match (average x-location of defensive actions)
5. First look: how do these metrics vary by manager era?

The modelling (regression, press–line relationship) happens in the next steps once we have clean per-match features.

In [ ]:
import pandas as pd
import numpy as np
from statsbombpy import sb
from tqdm import tqdm

pd.set_option('display.max_columns', None)

## 1. Load all Barcelona La Liga matches and assign manager eras

We label each match by the manager in charge. Tenure dates are approximate — we flag any boundary matches for manual inspection.

In [ ]:
# Manager eras at Barcelona — approximate start dates
# Sources: Wikipedia / Transfermarkt
MANAGER_ERAS = [
    ('Rijkaard',   '2003-07-01', '2008-06-30'),
    ('Guardiola',  '2008-07-01', '2012-06-30'),
    ('Vilanova',   '2012-07-01', '2013-07-18'),
    ('Martino',    '2013-07-19', '2014-05-31'),
    ('Luis Enrique','2014-06-01', '2017-06-30'),
    ('Valverde',   '2017-07-01', '2020-01-13'),
    ('Setien',     '2020-01-14', '2020-08-17'),
    ('Koeman',     '2020-08-18', '2021-10-27'),
]

def assign_manager(date_str):
    d = pd.Timestamp(date_str)
    for name, start, end in MANAGER_ERAS:
        if pd.Timestamp(start) <= d <= pd.Timestamp(end):
            return name
    return 'Unknown'

# Load all Barca La Liga matches (reusing logic from notebook 01)
COMPETITION_ID = 11
BARCA = 'Barcelona'

competitions = sb.competitions()
la_liga = (
    competitions[competitions['competition_name'] == 'La Liga']
    [['season_id', 'season_name']]
    .drop_duplicates()
)

all_matches = []
for _, row in la_liga.iterrows():
    m = sb.matches(competition_id=COMPETITION_ID, season_id=row['season_id'])
    m['season_name'] = row['season_name']
    all_matches.append(m)

matches = pd.concat(all_matches, ignore_index=True)
barca_matches = matches[
    (matches['home_team'] == BARCA) | (matches['away_team'] == BARCA)
].copy()

barca_matches['barca_venue'] = barca_matches['home_team'].apply(
    lambda t: 'home' if t == BARCA else 'away'
)
barca_matches['opponent'] = barca_matches.apply(
    lambda r: r['away_team'] if r['home_team'] == BARCA else r['home_team'], axis=1
)
barca_matches['manager'] = barca_matches['match_date'].apply(assign_manager)

print(barca_matches.groupby('manager')['match_id'].count().rename('matches'))

## 2. Extract per-match metrics from event data

For each match we compute three numbers:

- **`offsides_drawn`**: passes by the opponent flagged as `Pass Offside` — each one is a trap success.
- **`ppda`**: Passes Per Defensive Action — the standard pressing intensity metric. Lower = more intense press. Computed as: (opponent passes in the final 60% of the pitch) / (Barcelona defensive actions in the same zone).
- **`def_line_height`**: mean x-coordinate of Barcelona's defensive actions (tackles, interceptions, clearances). StatsBomb pitches run 0–120 (length). Higher = more advanced line.

This loop is slow (~1–2 min for all matches). Output is saved to `data/processed/`.

In [ ]:
DEFENSIVE_ACTIONS = ['Tackle', 'Interception', 'Clearance', 'Block']
# PPDA: pressing zone = opponent's 60 % of pitch (x > 48 on a 120-length pitch)
PPDA_ZONE_X = 48.0

def extract_match_metrics(match_id: int, barca_team: str) -> dict:
    events = sb.events(match_id=match_id)
    opponent = [t for t in events['team'].unique() if t != barca_team][0]

    # --- offsides drawn (opponent passes flagged offside) ---
    offsides = (
        (events['type'] == 'Pass') &
        (events['pass_outcome'] == 'Pass Offside') &
        (events['team'] == opponent)
    ).sum()

    # --- PPDA ---
    # Opponent passes in pressing zone
    opp_passes = events[
        (events['type'] == 'Pass') &
        (events['team'] == opponent)
    ].copy()
    opp_passes_in_zone = opp_passes[
        opp_passes['location'].apply(
            lambda loc: isinstance(loc, list) and loc[0] > PPDA_ZONE_X
        )
    ]
    # Barcelona defensive actions in same zone
    barca_def = events[
        (events['type'].isin(DEFENSIVE_ACTIONS)) &
        (events['team'] == barca_team)
    ].copy()
    barca_def_in_zone = barca_def[
        barca_def['location'].apply(
            lambda loc: isinstance(loc, list) and loc[0] > PPDA_ZONE_X
        )
    ]
    n_def = len(barca_def_in_zone)
    ppda = len(opp_passes_in_zone) / n_def if n_def > 0 else np.nan

    # --- Defensive line height ---
    def_x_values = barca_def['location'].apply(
        lambda loc: loc[0] if isinstance(loc, list) else np.nan
    ).dropna()
    def_line_height = def_x_values.mean() if len(def_x_values) else np.nan

    return {
        'match_id': match_id,
        'offsides_drawn': int(offsides),
        'ppda': ppda,
        'def_line_height': def_line_height,
        'n_opp_passes_zone': len(opp_passes_in_zone),
        'n_barca_def_zone': n_def,
    }


rows = []
for _, row in tqdm(barca_matches.iterrows(), total=len(barca_matches), desc='Matches'):
    metrics = extract_match_metrics(row['match_id'], BARCA)
    rows.append(metrics)

match_metrics = pd.DataFrame(rows)

## 3. Merge and save

In [ ]:
barca_data = barca_matches.merge(match_metrics, on='match_id')

# Save for use in subsequent notebooks
barca_data.to_csv('../data/processed/messi_era_match_metrics.csv', index=False)

print(f"Saved {len(barca_data)} matches")
barca_data[[
    'season_name', 'manager', 'barca_venue', 'opponent',
    'offsides_drawn', 'ppda', 'def_line_height'
]].head(10)

## 4. First look: metrics by manager era

Before any modelling, we simply look at how these three metrics vary across managerial eras. This is purely descriptive — we are not making causal claims about manager choices here.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

era_order = [e[0] for e in MANAGER_ERAS if e[0] in barca_data['manager'].unique()]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Barcelona La Liga — Tactical Metrics by Manager Era", fontsize=13)

metrics = [
    ('offsides_drawn', 'Offsides drawn per match',  False),
    ('ppda',           'PPDA (lower = more pressing)', False),
    ('def_line_height','Defensive line height (x, 0–120)', False),
]

for ax, (col, title, _) in zip(axes, metrics):
    sns.boxplot(
        data=barca_data, x='manager', y=col,
        order=era_order, ax=ax
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('../reports/figures/01_metrics_by_era.png', dpi=150, bbox_inches='tight')
plt.show()

## What comes next

We now have clean per-match values for offsides drawn, PPDA, and defensive line height across the full Messi era.

The next analytical steps (to be developed from this point) are:

1. **Press–line correlation**: does PPDA predict offsides drawn? Do matches with low pressing (high PPDA) and a high defensive line concede more?
2. **Identify trap failures**: extract event sequences where a pass beat the line and led directly to a shot — label the pass technique (through ball, long ball, etc.).
3. **Model**: a simple OLS or mixed-effects regression of `offsides_drawn ~ ppda + def_line_height + opponent_strength + barca_venue + manager`.
4. **Validate**: do the metrics behave sensibly for matches we know were tactically extreme (e.g., Guardiola's 5-0 Clásico, Setién's 8-2 Bayern loss)?